In [ ]:
# 1 - Import thư viện và module cần thiết

from src import reset_invoice_no, ready_order, staff_rename, stage_0
from core import (
    load_and_normalize,     smart_data_pipeline, 
    anonymize_customer_pii, process_product_master,
    insert_hash_sku_imei,   anonymize_sales_product, 
    mimic_price_history_and_payments, 
    join_sales_traffic,     bf_fill)
from contextlib import redirect_stdout
from dotenv import load_dotenv
from pathlib import Path
import pandas as pd
import json
import sys
import os
import io

BASE_DIR = Path(r"D:\Python\Dynamic_Dataframe")

f = io.StringIO()

# 2 - Lấy Salt và Random Seed từ file .env
load_dotenv(dotenv_path=BASE_DIR / 'config' / 'SECRET.env')
if os.getenv('CUST_SALT_KEY') is None:
    print("Vui lòng kiểm tra file SECRET.env")
    exit(1)
cust_salt = int(os.getenv('CUST_SALT_KEY'))
prod_salt = int(os.getenv('PRODUCT_SALT_KEY'))
ran_seed  = int(os.getenv('RANDOM_SEED'))
traffic_scale = [float(x) for x in os.getenv('TRAFFIC_SCALE').split(",")]

# 3 - Load JSON scale_dict cho price scaling
try:
    with open(BASE_DIR / 'config' / 'DYNAMIC_PRICE.json', 'r') as js:
        scale_dict = json.load(js)
except Exception as e:
    print(f"Vui lòng kiểm tra file DYNAMIC_PRICE.json: {e}")
    exit(1)

# 4 - Path đến source CSV và config xử lý
csv_2024     = BASE_DIR / 'CSV_read_only' / 'SALES_APPLE_2024_CLEAN.csv'
csv_2025     = BASE_DIR / 'CSV_read_only' / 'SALES_APPLE_2025_DIRTY.csv'
product_info = BASE_DIR / 'CSV_read_only' / 'UPDATED_ALL_PRICE_JAMES.csv'
anonym_price = BASE_DIR / 'data_output'   / 'Anonym_Price.csv'
traffic_path = BASE_DIR / 'CSV_read_only' / 'APPLE_2024_2026_FAKE_TRAFFIC.parquet'

    # config chung cho pipeline
config = {
    'payment_cols': ['cash', 'card', 'payoo', 'banking', 'mkt', 'vnpay', 'trade_in'],
    'disc_cols'   : ['disc_percent', 'disc_amount'],
    'date_anchor' : 'invoice',
    'date_pocket' : 'date',
    'drop_col'    : ['vat', 'note'],
    'anonymous'   : ['phone', 'name', 'email',]
    }

    # Tham số cho hàm ready_order(), cleaning và sắp xếp lại cột cho df_ready
_orders = ['date', 'invoice', 'sa', 'sku', 'imei_sn', 'cat', 
        'detail_sub_lob', 'product_name', 'price', 'qty', 
        'ins_stt', 'ins_fee', 'disc_percent', 'disc_amount', 
        'revenue', 'cash', 'card', 'qr_code']
trash_cols = ['ean', 'fill_date', 'no_payment']
    

# product_master = process_product_master( # NOTE: Hàm này có thể chạy độc lập nếu chỉ cần xử lý product master, export CSV nếu muốn
#     product_info=product_info, 
#     prod_salt=prod_salt, 
#     ran_seed=ran_seed, 
#     scale_dict=scale_dict)

# 5 - Chạy pipeline xử lý dữ liệu -> df_master
with redirect_stdout(f):
    raw_df = load_and_normalize(csv_2024, csv_2025, config)
    df = (raw_df
        .pipe(smart_data_pipeline, 
            config=config)

        .pipe(anonymize_customer_pii, 
            config=config, 
            cust_salt=cust_salt)

        .pipe(insert_hash_sku_imei, 
            cust_salt=cust_salt, 
            anonym_path=anonym_price)

        .pipe(anonymize_sales_product, 
            anonym_path=anonym_price)

        .pipe(mimic_price_history_and_payments, 
            config=config)
    )

    df_ready = (df
        .pipe(reset_invoice_no)

        .pipe(ready_order, 
            trash_cols=trash_cols, 
            prefer_order=_orders)

        .pipe(staff_rename)
    )

    _cust_anc = 'invoice'
    _cust_cols = ['id', 'name', 'email']

    df_master = (df_ready
        .pipe(join_sales_traffic, 
            traffic_path=traffic_path)
        .pipe(bf_fill, 
            _anchor=_cust_anc, 
            _target_cols=_cust_cols))


In [ ]:
def filter_data(df, month_step=None, categories=None, date_col='date', cat_col='cat'):
    """
    Lọc dataframe theo khoảng tháng gần nhất và danh mục.
    month_step: 1, 3, 6, 12 hoặc None (All time)
    categories: list các category hoặc None (Lấy tất cả)
    """
    query_parts = []
    params = {}

    # 1. Xử lý logic Thời gian
    if month_step is not None:
        target_date = pd.Timestamp.now() - pd.DateOffset(months=month_step)
        query_parts.append(f"`{date_col}` >= @target_date")
        params['target_date'] = target_date

    # 2. Xử lý logic Category
    if categories and len(categories) > 0:
        query_parts.append(f"`{cat_col}` in @categories")
        params['categories'] = categories
    print(f"DEBUG: Giá trị của ket_qua là: {query_parts}")
    print(f"DEBUG: Giá trị của ket_qua là: {params}")
    # 3. Kết hợp và thực thi
    if not query_parts:
        return df
        
    final_query = " and ".join(query_parts)
    print(f'ok kok {final_query}')
    return df.query(final_query, local_dict=params)




df_master = filter_data(df_master)
df_master = df_master.query("", local_dict={})

DEBUG: Giá trị của ket_qua là: []
DEBUG: Giá trị của ket_qua là: {}


ValueError: expr cannot be an empty string